# MarketPulse — A/B Testing: Checkout Experience Experiment

Analyzes the simulated checkout-redesign experiment (`marketpulse_experiments`): control vs. treatment conversion rate, using a two-proportion z-test, with confidence interval and effect size, ending in a ship/no-ship recommendation.

**Note:** this experiment is fully synthetic — customers were randomly assigned to control/treatment with a deliberate uplift built into the generator (documented in NOTES.md). This notebook demonstrates correct A/B testing methodology on that simulated data, not a real organic experiment result.

In [0]:
import pandas as pd
import numpy as np
from scipy import stats

# Load and Inspect
experiments_pd = spark.sql("SELECT * FROM marketpulse_experiments").toPandas()

print(f"Total customers in experiment: {experiments_pd.shape[0]}")
print(experiments_pd["variant"].value_counts())
print(f"\nOverall conversion rate: {experiments_pd['conversion'].mean():.4f}")


Total customers in experiment: 20000
variant
treatment    10093
control       9907
Name: count, dtype: int64

Overall conversion rate: 0.1144


## 2. Conversion Rate by Variant

Splits the experiment data by variant and computes each group's conversion rate — the primary metric for this experiment.

In [0]:
conversion_summary = (
    experiments_pd.groupby("variant")["conversion"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "conversions", "count": "total", "mean": "conversion_rate"})
)
conversion_summary["conversion_rate_pct"] = (conversion_summary["conversion_rate"] * 100).round(2)

print(conversion_summary)

           conversions  total  conversion_rate  conversion_rate_pct
variant                                                            
control            974   9907         0.098314                 9.83
treatment         1315  10093         0.130288                13.03


## 3. Two-Proportion Z-Test — Primary Metric (Conversion Rate)

**H0:** Control and treatment have the same true conversion rate.
**H1:** Control and treatment have different true conversion rates.

A two-proportion z-test is the correct test here — unlike the earlier tests in this project, we're comparing the difference between two *proportions* (conversion rate), not two *means* of a continuous variable. This is the standard test used in real A/B testing for exactly this kind of binary outcome (converted / did not convert).

Installs `statsmodels`, used below for the two-proportion z-test and Wilson confidence intervals (not pre-installed in this Databricks environment by default).

In [0]:
%pip install statsmodels

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control_conversions = conversion_summary.loc["control", "conversions"]
control_total = conversion_summary.loc["control", "total"]
treatment_conversions = conversion_summary.loc["treatment", "conversions"]
treatment_total = conversion_summary.loc["treatment", "total"]

# Two-proportion z-test
count = np.array([treatment_conversions, control_conversions])
nobs = np.array([treatment_total, control_total])
z_stat, p_value = proportions_ztest(count, nobs, alternative="two-sided")

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

# Confidence intervals for each group's conversion rate (95%)
ci_treatment = proportion_confint(treatment_conversions, treatment_total, alpha=0.05, method="wilson")
ci_control = proportion_confint(control_conversions, control_total, alpha=0.05, method="wilson")
print(f"\nTreatment 95% CI: [{ci_treatment[0]*100:.2f}%, {ci_treatment[1]*100:.2f}%]")
print(f"Control 95% CI: [{ci_control[0]*100:.2f}%, {ci_control[1]*100:.2f}%]")

# Absolute and relative uplift
abs_uplift = conversion_summary.loc["treatment", "conversion_rate"] - conversion_summary.loc["control", "conversion_rate"]
rel_uplift = abs_uplift / conversion_summary.loc["control", "conversion_rate"]
print(f"\nAbsolute uplift: {abs_uplift*100:.2f} percentage points")
print(f"Relative uplift: {rel_uplift*100:.2f}%")

# Effect size for two proportions: Cohen's h
p1 = conversion_summary.loc["treatment", "conversion_rate"]
p2 = conversion_summary.loc["control", "conversion_rate"]
cohens_h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))
print(f"Cohen's h (effect size for proportions): {cohens_h:.4f}")

Z-statistic: 7.1015
P-value: 0.000000

Treatment 95% CI: [12.39%, 13.70%]
Control 95% CI: [9.26%, 10.43%]

Absolute uplift: 3.20 percentage points
Relative uplift: 32.52%
Cohen's h (effect size for proportions): 0.1007


## 4. Secondary Metric — Revenue Per User

Checks whether the conversion uplift translates into higher revenue, or whether treatment converts more customers at a lower average value, potentially offsetting the gain. This directly tests the condition attached to the ship recommendation above.

In [0]:
revenue_summary = (
    experiments_pd.groupby("variant")["revenue"]
    .agg(["sum", "mean", "count"])
    .rename(columns={"sum": "total_revenue", "mean": "revenue_per_user", "count": "total_customers"})
)
print(revenue_summary)

# Revenue per user = total revenue / total customers exposed (not just converters) —
# this is the correct metric for "did this variant make more money per person shown it",
# not average order value among only those who converted
print(f"\nRevenue per user — control: ₹{revenue_summary.loc['control', 'revenue_per_user']:.2f}")
print(f"Revenue per user — treatment: ₹{revenue_summary.loc['treatment', 'revenue_per_user']:.2f}")

rev_uplift_pct = (revenue_summary.loc['treatment', 'revenue_per_user'] / revenue_summary.loc['control', 'revenue_per_user'] - 1) * 100
print(f"Revenue per user uplift: {rev_uplift_pct:.2f}%")

# t-test on revenue per user (comparing the full exposed population, including non-converters at ₹0)
from scipy import stats as scipy_stats
t_stat, p_value_rev = scipy_stats.ttest_ind(
    experiments_pd[experiments_pd["variant"]=="treatment"]["revenue"],
    experiments_pd[experiments_pd["variant"]=="control"]["revenue"],
    equal_var=False
)
print(f"\nT-test on revenue per user: t={t_stat:.2f}, p-value={p_value_rev:.6f}")

           total_revenue  revenue_per_user  total_customers
variant                                                    
control        116958.63         11.805656             9907
treatment      160970.77         15.948754            10093

Revenue per user — control: ₹11.81
Revenue per user — treatment: ₹15.95
Revenue per user uplift: 35.09%

T-test on revenue per user: t=6.08, p-value=0.000000


## 4. Secondary Metric — Average Order Value by Variant

Checks whether the treatment's conversion uplift came at the cost of lower spend per converting customer — a conversion increase paired with a revenue drop would change the ship recommendation.

In [0]:
converted = experiments_pd[experiments_pd["conversion"] == True]

aov_summary = converted.groupby("variant")["revenue"].agg(["mean", "median", "count"])
print(aov_summary)

control_rev = converted[converted["variant"] == "control"]["revenue"]
treatment_rev = converted[converted["variant"] == "treatment"]["revenue"]

t_stat, p_value = stats.ttest_ind(treatment_rev, control_rev, equal_var=False)
print(f"\nT-test on revenue (converted customers only): t={t_stat:.4f}, p-value={p_value:.6f}")

# Revenue per user (RPU) — accounts for both conversion rate AND spend, the real bottom-line metric
rpu_control = converted[converted["variant"]=="control"]["revenue"].sum() / conversion_summary.loc["control","total"]
rpu_treatment = converted[converted["variant"]=="treatment"]["revenue"].sum() / conversion_summary.loc["treatment","total"]
print(f"\nRevenue per user — control: {rpu_control:.2f}")
print(f"Revenue per user — treatment: {rpu_treatment:.2f}")

                 mean  median  count
variant                             
control    120.080729  101.89    974
treatment  122.411232  105.29   1315

T-test on revenue (converted customers only): t=0.6444, p-value=0.519405

Revenue per user — control: 11.81
Revenue per user — treatment: 15.95


## Summary

**Experiment:** Simulated checkout experience redesign, tested on 20,000 customers (control: 9,907, treatment: 10,093).

**Primary metric — Conversion rate:** Control 9.83%, Treatment 13.03%. Two-proportion z-test: Z=7.10, p≈0.000000. Statistically significant, non-overlapping 95% CIs. Cohen's h=0.1007 (small standardized effect despite the large relative uplift of 32.52% — a large sample size makes even a modest true effect highly significant).

**Secondary metric — Revenue impact:** No significant difference in average order value between converters (p=0.519) — the conversion lift does not come at the cost of lower spend per customer. Revenue per user: control ₹11.81, treatment ₹15.95 (~35% higher).

**Recommendation:** Ship the treatment. The effect is statistically robust and translates into a meaningful revenue improvement with no evidence of an offsetting cost, though the standardized effect size suggests a modest rather than dramatic real-world impact.

**Note:** this experiment is fully synthetic, with a deliberate conversion uplift built into the data generator (documented in NOTES.md). This notebook demonstrates correct end-to-end A/B testing methodology — hypothesis, primary and secondary metrics, proper test selection, confidence intervals, effect size, and a business recommendation — on that simulated data.